In [3]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Load dữ liệu đã xử lý
PROCESSED_DIR = '../dataset/processed/'
print("Đang load dữ liệu...")
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'master_data.parquet'))

print("Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...")

# Lấy mẫu 100 sản phẩm đầu tiên để chạy cho nhanh (Phục vụ vẽ báo cáo)
sample_items = df['item_id'].unique()[:100]
df_run = df[df['item_id'].isin(sample_items)]

def classify_demand(group):
    """Hàm tính ADI, CV2 và phân loại cho từng sản phẩm"""
    demand = group['demand'].values
    
    # Nếu không bán được cái nào trong lịch sử
    if demand.sum() == 0:
        return pd.Series({'ADI': np.nan, 'CV2': np.nan, 'Category': 'No Sales'})
        
    # Tìm index của ngày đầu tiên bắt đầu bán hàng
    non_zero_indices = np.where(demand > 0)[0]
    first_sale_idx = non_zero_indices[0]
    active_demand = demand[first_sale_idx:]
    
    # 1. Tính ADI (Số ngày / Số ngày có khách mua)
    days_with_sales = len(active_demand[active_demand > 0])
    if days_with_sales == 0:
        adi = np.nan
    else:
        adi = len(active_demand) / days_with_sales
        
    # 2. Tính CV2 ((Độ lệch chuẩn / Số trung bình)^2)
    mean_demand = np.mean(active_demand)
    if mean_demand == 0:
        cv2 = 0
    else:
        std_demand = np.std(active_demand)
        cv2 = (std_demand / mean_demand) ** 2
        
    # 3. Phân loại theo Ma trận Syntetos-Boylan
    if adi <= 1.32 and cv2 <= 0.49:
        category = 'Smooth (Đều đặn)'
    elif adi <= 1.32 and cv2 > 0.49:
        category = 'Erratic (Thất thường)'
    elif adi > 1.32 and cv2 <= 0.49:
        category = 'Intermittent (Ngắt quãng)'
    else:
        category = 'Lumpy (Cục bộ)'
        
    return pd.Series({'ADI': round(adi, 2), 'CV2': round(cv2, 2), 'Category': category})

# 2. Áp dụng hàm phân loại
classification_df = df_run.groupby(['item_id', 'store_id']).apply(classify_demand).reset_index()
classification_df = classification_df[classification_df['Category'] != 'No Sales']

print("\n✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:")
print(classification_df['Category'].value_counts())

print("\nBảng phân loại chi tiết (5 dòng đầu):")
display(classification_df.head())

Đang load dữ liệu...
Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...

✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:
Category
Lumpy (Cục bộ)           196
Erratic (Thất thường)      4
Name: count, dtype: int64

Bảng phân loại chi tiết (5 dòng đầu):


,item_id,store_id,ADI,CV2,Category
0,FOODS_1_001,CA_1,NaN,NaN,NaN
1,FOODS_1_001,CA_2,NaN,NaN,NaN
2,FOODS_1_002,CA_1,NaN,NaN,NaN
3,FOODS_1_002,CA_2,NaN,NaN,NaN
4,FOODS_1_003,CA_1,NaN,NaN,NaN
